# BigData-Viz · Demonstrația pilot

Comparăm agregarea exactă cu estimările eșantioanelor Bernoulli pentru două întrebări despre cursele NYC Yellow Taxi. Graficele pilot, timpii și erorile provin din aceeași rulare A3 salvată.

Selectează kernelul `.venv` din VS Code, apoi folosește **Restart Kernel → Run All**. Schimbă `RUN_ID` dacă rezultatele tale se află în alt director de rulare.

In [ ]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import Markdown, display

cwd = Path.cwd()
project = next(
    (path for path in (cwd, *cwd.parents) if (path / "src/render.py").is_file()),
    None,
)
if project is None:
    raise FileNotFoundError("Deschide notebookul din proiectul BigData-Viz.")
sys.path.insert(0, str(project))

from src.config import RUNS_DIR
from src.render import build_pilot_figures, load_pilot_artifacts

RUN_ID = "pilot_01"
run_dir = RUNS_DIR / RUN_ID
required = ("run_manifest.json", "results.csv", "timings.csv", "errors.csv")
missing = [name for name in required if not (run_dir / name).is_file()]
if missing:
    raise FileNotFoundError(f"Lipsesc fișierele din {run_dir}: {', '.join(missing)}")

rows, manifest, errors = load_pilot_artifacts(run_dir)
protocol = manifest["protocol"]
period = manifest["period"]
period_text = f"[{period['start'][:10]}, {period['end'][:10]})"
data_kind = "date reale" if manifest["data_kind"] == "real" else "fixture de test"
display(Markdown(
    f"**Rulare:** `{manifest['run_id']}` · **Dataset:** `{manifest['dataset_id']}` "
    f"({data_kind}) · **Perioadă:** `{period_text}` · "
    f"**Seed:** `{protocol['seeds'][0]}` · **Repetări:** `{protocol['repeats']}`"
))

## Exemplu de agregare exactă

Rulăm o singură interogare pe fixture-ul comun, ca să vedem cum se formează seriile pe 24 de ore. Acest exemplu este independent de valorile pilot afișate mai jos.

In [ ]:
import duckdb

from src.config import get_dataset
from src.data import register_trips
from src.queries import query_counts

jan = get_dataset("jan")
fixture = project / "tests/fixtures/trips.csv"
with duckdb.connect() as con:
    register_trips(con, [fixture], jan.start, jan.end)
    exact_fixture = query_counts(con, "hours", "trips").fetchall()
fixture_hours = pd.DataFrame(exact_fixture, columns=["Ora", "Curse"])
display(fixture_hours[fixture_hours["Curse"] > 0].reset_index(drop=True))

## Graficul pe ore și distribuția distanțelor

Schimbă procentul în selectorul fiecărui grafic. Graficul pe ore afișează ponderea fiecărei ore raportată la totalul propriu al seriei exacte sau estimate. Eroarea din titlu este eroarea normalizată pe numărul total de curse, calculată din numărătorile pe ore, nu o abatere între procentele orare. Graficul distanțelor afișează proporția curselor din fiecare interval, normalizată separat pentru fiecare serie. Estimările și erorile provin din `results.csv` și `errors.csv`, validate împreună cu manifestul de rulare.


In [ ]:
figures = build_pilot_figures(rows, manifest, errors=errors)
figures["hours"].show(renderer="plotly_mimetype")
figures["distance"].show(renderer="plotly_mimetype")

## Timpi și erori din rularea salvată

Mediana descrie timpul interogării. Construirea eșantioanelor este măsurată separat. Pentru ore folosim abaterea absolută normalizată, iar pentru distanțe folosim distanța de variație totală.

In [ ]:
identity = ["run_id", "dataset_id", "config_version"]
key_types = {"config_version": str, "seed": "Int64"}
timings = pd.read_csv(run_dir / "timings.csv", dtype=key_types)
errors = pd.DataFrame(errors)
errors["seed"] = errors["seed"].astype("Int64")
errors["sample_rate"] = pd.to_numeric(errors["sample_rate"], errors="raise")
errors["value"] = pd.to_numeric(errors["value"], errors="raise")
for frame, count in ((timings, "measured_queries"), (errors, "error_rows")):
    if len(frame) != manifest[count] or any(
        not frame[field].eq(manifest[field]).all() for field in identity
    ):
        raise ValueError("Fișierele cu timpi și erori nu corespund manifestului.")

case_keys = ["question", "method", "sample_rate", "seed"]
medians = timings.groupby(case_keys, dropna=False)["elapsed_seconds"].median() * 1000
summary = medians.rename("median_ms").reset_index().merge(
    errors[case_keys + ["metric", "value", "status"]],
    on=case_keys, how="left", validate="one_to_one",
)
summary["error_pct"] = summary.pop("value") * 100
display(summary.round({"median_ms": 3, "error_pct": 3}))

sample_build = pd.DataFrame([
    {"sample_rate": item["metadata"]["sample_rate"],
     "row_count": item["metadata"]["row_count"],
     "build_ms": item["metadata"]["build_seconds"] * 1000}
    for item in manifest["samples"]
])
display(sample_build.round({"build_ms": 3}))

## Interpretare și limite

Compară viteza interogărilor cu abaterea vizuală și numerică. Dacă eșantionul trebuie construit pentru o singură interogare, ia în calcul și costul din al doilea tabel. Pentru un eșantion gol, variația totală a distanțelor este nedefinită.

Pilotul testează o perioadă, un singur seed și trei repetări de timp pe caz. Fișierele HTML independente se obțin cu `python -m src.render --run-id pilot_01` și se pot deschide offline.